# 1. Batch Correction of RNA Embeddings

Apply batch correction methods (Harmony + DANN) to Foundation Model RNA embeddings.

**Пайплайн:** `0.eda_data_prep` → **`1.batch_correction`** → `2.metrics_modeling_stress_test`

**Вход:** `data/interim/prepared.adata.zarr` (из notebook 0)
**Выход:** `data/interim/corrected.adata.zarr`

**Steps:**
1. Load prepared AnnData (from notebook 0)
2. Baseline UMAP (before correction)
3. Harmony stage-1 + BackTrack stage-2
4. Comparison & save corrected AnnData

## environment

In [ ]:
# # !rm -rf batchcor_rna_emb/
# !git clone https://github.com/onion-42/batchcor-rna-embeds.git

# # or
# # git pull origin main

# %cd batchcor-rna-embeds

# !pip install -q uv
# !uv pip install --system -e .   # editable install из pyproject.toml
# !pip install -q --upgrade ipython ipykernel

Cloning into 'batchcor-rna-embeds'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 64 (delta 16), reused 62 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 69.57 KiB | 1.42 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/batchcor-rna-embeds
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 51.2 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
  × No solution found when resolving dependencies:                                  
  ╰─▶ Because imp was not found in the package registry and
      batchcor-rna-emb==0.0.1 depends on imp, we can conclude that
      batchcor-rna-emb==0.0.1 cannot be used.
      And because only batchcor-rna-emb==0.0.1 is available and you
      require batchcor-rna-emb, we can conclude that your requirements are
      unsatisfiable.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.3/627.3 kB 11.3 MB/s e

In [18]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from loguru import logger
from pathlib import Path

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    SEED,
    BATCH_COL,
    DIAGNOSIS_COL,
    SPLIT_PREFIX,
    TARGET_PREFIX,
    HARMONY_SUFFIX,
    DANN_SUFFIX,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr

# Loguru setup
set_logger(level="INFO")

# Reproducibility
np.random.seed(SEED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# Scanpy / UMAP figure params
sc.settings.figdir = str(FIGURES_DIR)
sc.set_figure_params(dpi=150, dpi_save=300, figsize=(5, 5))

UMAP_DIST = 0.4
UMAP_SPREAD = 1.0

# Ensure output directories exist
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Prepared AnnData (from notebook 0)

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import anndata

/home/nadezhdalukashevich/venvs/immesc_pred_venvs_cpu/lib/python3.10/site-packages/anndata/_core/anndata.py:22: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 1.22.4)
  from scipy import sparse


In [5]:
import zarr

In [6]:
path = 'data/raw/NSCLC_Tissue_ICI_Pred.adata.zarr'
root = zarr.open(path, mode='a')
zarr.consolidate_metadata(root.store)

<zarr.hierarchy.Group '/'>

In [ ]:
group = root['obs']
attrs = dict(group.attrs)

current_columns = list(attrs.get("column-order", []))
for column_name in current_columns:
        current_columns.append(column_name)
        has_changes = True

group.attrs["column-order"] = current_columns

In [10]:
list(root['obs'])

['Age',
 'Cohort',
 'Diagnosis',
 'Diagnosis_long',
 'F_Therapy_Eval',
 'Gender',
 'Kassandra_B_cells',
 'Kassandra_CD4_T_cells',
 'Kassandra_CD8_T_cells',
 'Kassandra_CD8_T_cells_PD1_high',
 'Kassandra_CD8_T_cells_PD1_low',
 'Kassandra_Central_memory_T_helpers',
 'Kassandra_Class_switched_memory_B_cells',
 'Kassandra_Cytotoxic_NK_cells',
 'Kassandra_Effector_memory_T_helpers',
 'Kassandra_Endothelium',
 'Kassandra_Fibroblasts',
 'Kassandra_Lymphoid_cells',
 'Kassandra_Macrophages',
 'Kassandra_Macrophages_CXCL9',
 'Kassandra_Macrophages_Double_negative',
 'Kassandra_Macrophages_M1',
 'Kassandra_Macrophages_M2',
 'Kassandra_Macrophages_SPP1',
 'Kassandra_Mature_B_cells',
 'Kassandra_Memory_CD8_T_cells',
 'Kassandra_Memory_T_cells',
 'Kassandra_Myeloid_cells',
 'Kassandra_NK_cells',
 'Kassandra_Naive_B_cells',
 'Kassandra_Naive_CD8_T_cells',
 'Kassandra_Naive_T_cells',
 'Kassandra_Naive_T_helpers',
 'Kassandra_Neutrophils',
 'Kassandra_Non_switched_memory_B_cells',
 'Kassandra_Other',
 

In [7]:
anndata.read_zarr('data/raw/NSCLC_Tissue_ICI_Pred.adata.zarr')

KeyError: 'RND_USE'

In [19]:
DATA_RAW_DIR = Path('/content/drive/MyDrive/BG_Internship_group_7/data')

In [20]:
# Загрузка подготовленного AnnData
adata = load_cohort(DATA_RAW_DIR / "Melanoma_Tissue_ICI_Pred.adata.zarr")

logger.info("Loaded AnnData: {} samples x {} genes", adata.n_obs, adata.n_vars)
logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
logger.info("Diagnosis distribution:\n{}", adata.obs[DIAGNOSIS_COL].value_counts())
logger.info(".obsm keys: {}", list(adata.obsm.keys()))

KeyError: 'Path_Eurynome'

In [ ]:
# Определение ключа FM-эмбеддинга
emb_keys = [k for k in adata.obsm.keys() if k.startswith("FM_") and HARMONY_SUFFIX not in k and DANN_SUFFIX not in k]
logger.info("Available FM embedding keys: {}", emb_keys)

EMB_KEY = emb_keys[0] if emb_keys else None
logger.info("Using embedding key: '{}'", EMB_KEY)

In [ ]:
# Извлечение split из metadata (создано в notebook 0)
filtered_target_name = adata.uns.get("filtered_target_name", "Response_wo_SD")
split_col = f"{SPLIT_PREFIX}{filtered_target_name}"

if split_col not in adata.obs.columns:
    # Fallback: ищем любую Split_ колонку
    split_candidates = [c for c in adata.obs.columns if c.startswith("Split_")]
    if split_candidates:
        split_col = split_candidates[0]
        logger.warning("Ожидаемая '{}' не найдена, используем '{}'",
                       f"{SPLIT_PREFIX}{filtered_target_name}", split_col)
    else:
        raise KeyError("Не найдена колонка сплита — запустите сначала notebook 0")

# Унифицированная колонка split для Harmony BackTrack
adata.obs["split"] = adata.obs[split_col].astype(str)

adata_train = adata[adata.obs["split"] == "train"].copy()
adata_test = adata[adata.obs["split"] == "test"].copy()
logger.info("Split col: '{}' | Train: {}, Test: {}", split_col, adata_train.n_obs, adata_test.n_obs)

## 2. Baseline Visualization (Before Correction)

In [ ]:
# Compute UMAP on raw (uncorrected) embeddings — train only
sc.pp.neighbors(adata_train, use_rep=EMB_KEY, n_neighbors=30, metric="cosine")
sc.tl.umap(adata_train, min_dist=UMAP_DIST, spread=UMAP_SPREAD)

sc.pl.umap(adata_train, color=BATCH_COL, s=5, title="Raw Embeddings — Batch", show=True)
sc.pl.umap(adata_train, color=DIAGNOSIS_COL, s=5, title="Raw Embeddings — Diagnosis", show=True)

## 3. Harmony Batch Correction

**Stage-1:** Run Harmony on train embeddings to correct for batch effects.
**Stage-2 (BackTrack):** Project test onto frozen train UMAP via barycentric coordinates.

In [ ]:
from batchcor_rna_emb.batch_correction.harmony import run_harmony_stage1

# Stage-1: correct train embeddings
HARMONY_KEY = f"{EMB_KEY}{HARMONY_SUFFIX}"
corrected_train = run_harmony_stage1(adata_train, embedding_key=EMB_KEY, batch_col=BATCH_COL)
adata_train.obsm[HARMONY_KEY] = corrected_train
logger.info("Harmony stage-1 output stored in obsm['{}']", HARMONY_KEY)

In [ ]:
# Elbow plot: determine n_pcs for Harmony-corrected embeddings
harmony_pcs = adata_train.obsm[HARMONY_KEY]
variance = np.var(harmony_pcs, axis=0)
variance_sorted = np.sort(variance)[::-1][:80]

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(variance_sorted) + 1), variance_sorted, "o-", color="b", markersize=3)
plt.title("Elbow Plot: Harmony-corrected Embeddings")
plt.xlabel("Component")
plt.ylabel("Variance")
plt.grid(True)
plt.tight_layout()
plt.show()

# Determine n_pcs visually — adjust as needed
N_PCS_HARMONY = 42
logger.info("Using n_pcs={} for Harmony UMAP", N_PCS_HARMONY)

In [ ]:
# Sort Harmony embeddings by variance (descending) before UMAP
emb_matrix = adata_train.obsm[HARMONY_KEY]
variances = np.var(emb_matrix, axis=0)
desc_idx = np.argsort(variances)[::-1]
adata_train.obsm[HARMONY_KEY] = emb_matrix[:, desc_idx]

# Compute train UMAP on Harmony-corrected embeddings
sc.pp.neighbors(adata_train, use_rep=HARMONY_KEY, n_pcs=N_PCS_HARMONY, n_neighbors=30, metric="cosine")
sc.tl.umap(adata_train, min_dist=UMAP_DIST, spread=UMAP_SPREAD, key_added="UMAP_Harmony_")

sc.pl.embedding(adata_train, basis="UMAP_Harmony_", color=BATCH_COL, s=5, title="Harmony Stage-1 — Batch")
sc.pl.embedding(adata_train, basis="UMAP_Harmony_", color=DIAGNOSIS_COL, s=5, title="Harmony Stage-1 — Diagnosis")

In [ ]:
from batchcor_rna_emb.batch_correction.harmony import backtrack_harmony_integration, robustness_index

# BackTrack Harmony Integration: stage-2 + barycentric projection
combined, diag = backtrack_harmony_integration(
    adata_train, adata_test,
    embedding_key=EMB_KEY,
    batch_col=BATCH_COL,
    split_col="split",
    umap_train_key="UMAP_Harmony_",
    qc_neighbors=30,
    qc_metric="cosine",
    ood_q=0.995,
    ood_factor=1.0,
    ood_mode="stage2_knn",
    k_fallback=30,
)

logger.info("BackTrack diagnostics: {}", diag)

In [ ]:
# Post-Harmony UMAP: combined train+test
sc.pl.embedding(combined, basis="X_umap", color="split", s=5, title="BackTrack Harmony — Split")
sc.pl.embedding(combined, basis="X_umap", color=BATCH_COL, s=5, title="BackTrack Harmony — Batch")
sc.pl.embedding(combined, basis="X_umap", color=DIAGNOSIS_COL, s=5, title="BackTrack Harmony — Diagnosis")

In [ ]:
# QC: Robustness Index (RI = SO / (SO + OS))
ri_all, ri_details = robustness_index(
    combined,
    emb_key="X_umap",
    bio_key=DIAGNOSIS_COL,
    conf_key="combined_batch",
    k=50,
)
logger.info("Robustness Index: {:.3f}, details: {}", ri_all, ri_details)

## 4. DANN Batch Correction

**Domain Adversarial Neural Network (DANN)**:
- Encoder maps embeddings to batch-invariant latent space
- Gradient Reversal Layer (GRL) makes batch discriminator adversarial
- Optional bio classifier preserves diagnosis signal
- Progressive λ warmup over first 30% of epochs

Architecture: `Input → [512→BN→ReLU→Drop] → [256→BN→ReLU] → latent_dim`
Loss: `L_recon + λ_adv * L_adv - λ_bio * L_bio`

In [ ]:
from batchcor_rna_emb.batch_correction.dann import DANNCorrector, DANNConfig

dann_config = DANNConfig(
    latent_dim=256,
    n_epochs=100,
    batch_size=256,
    lr=1e-3,
    lambda_adv=1.0,
    lambda_bio=0.5,
    warmup_fraction=0.3,
    dropout=0.3,
    seed=SEED,
)

dann = DANNCorrector(config=dann_config)

# Fit on train embeddings
X_train_emb = np.asarray(adata_train.obsm[EMB_KEY], dtype=np.float32)
batch_labels = adata_train.obs[BATCH_COL].to_numpy()
bio_labels = adata_train.obs[DIAGNOSIS_COL].to_numpy() if DIAGNOSIS_COL in adata_train.obs.columns else None

dann.fit(X_train_emb, batch_labels, bio_labels)

In [ ]:
# Transform train and test embeddings
DANN_KEY = f"{EMB_KEY}{DANN_SUFFIX}"

adata_train.obsm[DANN_KEY] = dann.transform(X_train_emb)
X_test_emb = np.asarray(adata_test.obsm[EMB_KEY], dtype=np.float32)
adata_test.obsm[DANN_KEY] = dann.transform(X_test_emb)

logger.info("DANN latent stored in obsm['{}']: train={}, test={}",
            DANN_KEY, adata_train.obsm[DANN_KEY].shape, adata_test.obsm[DANN_KEY].shape)

In [ ]:
# Post-DANN UMAP
sc.pp.neighbors(adata_train, use_rep=DANN_KEY, n_neighbors=30, metric="cosine")
sc.tl.umap(adata_train, min_dist=UMAP_DIST, spread=UMAP_SPREAD, key_added="UMAP_DANN_")

sc.pl.embedding(adata_train, basis="UMAP_DANN_", color=BATCH_COL, s=5, title="DANN — Batch")
sc.pl.embedding(adata_train, basis="UMAP_DANN_", color=DIAGNOSIS_COL, s=5, title="DANN — Diagnosis")

In [ ]:
# DANN loss curves
from batchcor_rna_emb.visualization.plots import plot_loss_curves

history_dict = {
    "total": dann.history_.total,
    "recon": dann.history_.recon,
    "adversarial": dann.history_.adv,
    "bio": dann.history_.bio,
    "lambda": dann.history_.lambda_schedule,
}
plot_loss_curves(history_dict, title="DANN Training Loss", save_path=str(FIGURES_DIR / "dann_loss_curves.png"))

## 5. Comparison & Save

In [ ]:
# 2x3 UMAP comparison grid: Raw / Harmony / DANN × batch / diagnosis
from batchcor_rna_emb.visualization.plots import plot_umap_grid

# Build combined AnnData for each method (train only for fair comparison)
adata_raw = adata_train.copy()
adata_harmony = adata_train.copy()
adata_dann = adata_train.copy()

# Compute UMAP for raw (already done, in X_umap)
# Harmony UMAP already in UMAP_Harmony_
# DANN UMAP already in UMAP_DANN_

# Standardize to X_umap for the grid
adata_raw.obsm["X_umap"] = adata_raw.obsm["X_umap"]  # already set
adata_harmony.obsm["X_umap"] = adata_harmony.obsm["UMAP_Harmony_"]
adata_dann.obsm["X_umap"] = adata_dann.obsm["UMAP_DANN_"]

grid_adatas = {"Raw": adata_raw, "Harmony": adata_harmony, "DANN": adata_dann}
fig = plot_umap_grid(
    grid_adatas,
    color_cols=[BATCH_COL, DIAGNOSIS_COL],
    save_path=str(FIGURES_DIR / "umap_comparison_grid.png"),
)
plt.show()

In [ ]:
# Save corrected AnnData: recombine train + test with all obsm keys
adata_corrected = ad.concat([adata_train, adata_test], join="outer")
adata_corrected.obs["split"] = pd.Categorical(
    list(adata_train.obs["split"]) + list(adata_test.obs["split"])
)

# Перенос .uns metadata из исходного adata (notebook 0)
adata_corrected.uns.update(adata.uns)
adata_corrected.uns["embedding_key"] = EMB_KEY

output_path = DATA_INTERIM_DIR / "corrected.adata.zarr"
save_adata_zarr(adata_corrected, output_path)

logger.info("Saved corrected AnnData to {}", output_path)
logger.info(".obsm keys: {}", list(adata_corrected.obsm.keys()))
logger.info(".uns keys: {}", list(adata_corrected.uns.keys()))